In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/01-config

In [0]:
# COMMAND ----------

class Upserter:

    def __init__(self, merge_query, temp_view_name):
        self.merge_query = merge_query
        self.temp_view_name = temp_view_name

    def upsert(self, df_micro_batch, batch_id):

        df_micro_batch.createOrReplaceTempView(
            self.temp_view_name
        )

        spark.sql(self.merge_query)

### Classe `Upserter`

Cette classe permet d'exécuter un `MERGE` SQL sur chaque micro-batch du stream.

Le micro-batch est d'abord transformé en vue temporaire, puis utilisé comme source pour alimenter la table Gold.

`Micro-batch → Temp View → MERGE → Gold`

In [0]:
# COMMAND ----------

class Gold:

    def __init__(self, catalog="dev"):

        self.conf = Config()

        self.test_data_dir = (
            self.conf.base_dir_data + "/test_data"
        )

        self.checkpoint_base = (
            self.conf.base_dir_checkpoint
            + "/checkpoints"
        )

        self.catalog = catalog
        self.silver = self.conf.silver_schema
        self.gold = self.conf.gold_schema

        self.maxFilesPerTrigger = (
            self.conf.maxFilesPerTrigger
        )



    def upsert_workout_bpm_summary(
        self,
        once=True,
        processing_time="15 seconds",
        startingVersion=0
    ):

        from pyspark.sql import functions as F

        query = f"""
            MERGE INTO {self.catalog}.{self.gold}.workout_bpm_summary a
            USING workout_bpm_summary_delta b

            ON  a.user_id = b.user_id
            AND a.workout_id = b.workout_id
            AND a.session_id = b.session_id

            WHEN NOT MATCHED
            THEN INSERT *
        """

        data_upserter = Upserter(
            query,
            "workout_bpm_summary_delta"
        )

        df_users = spark.read.table(
            f"{self.catalog}.{self.silver}.user_bins"
        )

        df_delta = (
            spark.readStream
            .option(
                "startingVersion",
                startingVersion
            )
            .table(
                f"{self.catalog}.{self.silver}.workout_bpm"
            )
            .withWatermark(
                "end_time",
                "30 seconds"
            )
            .groupBy(
                "user_id",
                "workout_id",
                "session_id",
                "end_time"
            )
            .agg(
                F.min("heartrate").alias("min_bpm"),
                F.mean("heartrate").alias("avg_bpm"),
                F.max("heartrate").alias("max_bpm"),
                F.count("heartrate").alias("num_recordings")
            )
            .join(
                df_users,
                ["user_id"]
            )
            .select(
                "workout_id",
                "session_id",
                "user_id",
                "age",
                "gender",
                "city",
                "state",
                "min_bpm",
                "avg_bpm",
                "max_bpm",
                "num_recordings"
            )
        )

        stream_writer = (
            df_delta.writeStream
            .foreachBatch(
                data_upserter.upsert
            )
            .outputMode("append")
            .option(
                "checkpointLocation",
                f"{self.checkpoint_base}/workout_bpm_summary"
            )
            .queryName(
                "workout_bpm_summary_upsert_stream"
            )
        )

        if once:
            return (
                stream_writer
                .trigger(availableNow=True)
                .start()
            )

        return (
            stream_writer
            .trigger(
                processingTime=processing_time
            )
            .start()
        )
    #####################################################
    def upsert(
        self,
        once=True,
        processing_time="5 seconds"
    ):

        import time

        start = int(time.time())

        print(
            "\nExecuting Gold layer..."
        )

        self.upsert_workout_bpm_summary(
            once,
            processing_time
        )

        if once:
            for stream in spark.streams.active:
                stream.awaitTermination()

        print(
            f"Gold layer completed in "
            f"{int(time.time()) - start} seconds"
        )
    ###########################################################################################
    def assert_count(
    self,
    table_name,
    expected_count,
    filter_condition="true"):

        actual_count = (
            spark.read
            .table(
                f"{self.catalog}."
                f"{self.gold}."
                f"{table_name}"
            )
            .where(filter_condition)
            .count()
        )

        assert actual_count == expected_count, (
            f"Expected {expected_count:,}, "
            f"found {actual_count:,}"
        )

        print(
            f"{table_name}: "
            f"{actual_count:,} records - Success"
        )
    #############################################################################
    def assert_rows(
    self,
    location,
    table_name,
    sets):

        expected_rows = (
            spark.read
            .format("parquet")
            .load(
                f"{self.test_data_dir}/"
                f"{location}_{sets}.parquet"
            )
            .collect()
        )

        actual_rows = (
            spark.table(
                f"{self.catalog}."
                f"{self.gold}."
                f"{table_name}"
            )
            .collect()
        )

        assert expected_rows == actual_rows, (
            f"Expected data mismatches "
            f"with {table_name}"
        )

        print(
            f"{table_name}: expected data matches actual data - Success"
        )
    ##################################################
    def validate(self, sets=1):

        print(
            "\nValidating Gold layer..."
        )

        if sets > 1:
            self.assert_count(
                "workout_bpm_summary",
                2
            )

        print(
            "Gold layer validation completed."
        )

### Initialisation de la couche Gold

Le constructeur prépare les paramètres nécessaires au traitement Gold.

Il récupère notamment :

- le catalogue `dev`;
- le schéma Silver;
- le schéma Gold;
- le chemin des checkpoints;
- le chemin des données de test.

Cela évite de coder directement les chemins et noms de schémas dans chaque fonction.

### Création de `workout_bpm_summary`

Cette méthode construit la table Gold `workout_bpm_summary` à partir des données Silver.

Elle lit les mesures BPM de `workout_bpm`, les regroupe par séance, puis calcule :

- le BPM minimum ;
- le BPM moyen ;
- le BPM maximum ;
- le nombre total de mesures.

Le résultat est ensuite enrichi avec les informations utilisateur provenant de `user_bins`, comme l’âge, le genre, la ville et l’état.

Enfin, les nouvelles séances sont insérées dans la table Gold grâce à un `MERGE`.

Le flux est :

`workout_bpm + user_bins → agrégation des BPM → enrichissement utilisateur → MERGE → workout_bpm_summary`

### Lancement de la couche Gold

La méthode `upsert()` sert à lancer les traitements de la couche Gold.

Dans ce projet, elle exécute la création de `workout_bpm_summary` puis attend la fin du stream lorsque le mode `once=True` est utilisé.

### Validation du nombre de lignes

Cette fonction vérifie que la table Gold contient le nombre de lignes attendu.

Elle permet de détecter rapidement si le pipeline a produit trop ou pas assez de résultats.

### Validation des résultats

Cette fonction compare les données produites dans Gold avec un fichier Parquet contenant les résultats attendus.

Elle permet donc de vérifier le contenu réel des lignes, et pas seulement leur nombre.

`Résultat attendu ↔ Résultat obtenu`

### Validation finale de la couche Gold

La méthode `validate()` lance les contrôles nécessaires après l'exécution du pipeline Gold.

Elle permet de vérifier que les tables finales ont été correctement alimentées.